# EMIT / Sentinel-2 pairing, alignment, and tiling

## 1. Setup

In [ ]:
import os, getpass
os.environ["GH_USER"] = input("GitHub username: ")
os.environ["GH_TOKEN"] = getpass.getpass("GitHub PAT: ")

In [ ]:
import os, textwrap

askpass_path = "/tmp/gh_askpass.sh"
with open(askpass_path, "w") as f:
    f.write(textwrap.dedent("""\
        #!/bin/sh
        case "$1" in
          *Username*) echo "$GH_USER" ;;
          *Password*) echo "$GH_TOKEN" ;;
          *) echo "" ;;
        esac
    """))
os.chmod(askpass_path, 0o700)
os.environ["GIT_ASKPASS"] = askpass_path
os.environ["GIT_TERMINAL_PROMPT"] = "0"

!git clone https://github.com/incredet/HyperSpectralSuperResolution.git
%cd /content/HyperSpectralSuperResolution/

In [ ]:
!pip -q install -U py-tools-ds==0.24.1
!pip -q install -U arosics==1.13.2 geoarray==0.19.2
!pip install -q numpy xarray rasterio shapely pyproj matplotlib panel holoviews hvplot tqdm jupyter_bokeh pystac-client earthaccess netCDF4 spectral rioxarray hvplot POT
!pip install -q "git+https://github.com/EnSpec/hytools.git"
!apt-get install -y -q gdal-bin

In [ ]:
# Monkey-patch py_tools_ds to handle boolean arrays in warp_ndarray.
# py_tools_ds <= 0.24.1 passes the boolean mask directly to GDAL, which
# rejects non-numeric dtypes.  This wrapper converts booleans to uint8.
import importlib
import numpy as np
from py_tools_ds.geo.raster import reproject as reproj

importlib.reload(reproj)
_orig = reproj.warp_ndarray

def _warp_ndarray_mask_safe(ndarray, *args, **kwargs):
    for k in ("in_nodata", "out_nodata"):
        if k in kwargs and isinstance(kwargs[k], (bool, np.bool_)):
            kwargs[k] = int(kwargs[k])
    if isinstance(ndarray, np.ndarray) and ndarray.dtype == np.bool_:
        ndarray = ndarray.astype(np.uint8)
        kwargs.setdefault("in_nodata", 0)
        kwargs.setdefault("out_nodata", 0)
        kwargs.pop("out_dtype", None)
        kwargs.setdefault("rspAlg", "near")
    return _orig(ndarray, *args, **kwargs)

reproj.warp_ndarray = _warp_ndarray_mask_safe
print("warp_ndarray patched:", reproj.warp_ndarray is _warp_ndarray_mask_safe)

In [ ]:
import os, json, math
from pathlib import Path
from datetime import datetime, date, timezone, timedelta
from pprint import pprint

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import rasterio
from shapely.geometry import Point, box
from tqdm import tqdm
from rasterio.windows import Window

import panel as pn
import holoviews as hv
import hvplot.xarray
from pystac_client import Client
import earthaccess as ea

pn.extension("tabulator")
hv.extension("bokeh")

# ── project modules ──────────────────────────────────────────────────────
from data.pairing.pairs_utils import point_buffer_bbox
from data.pairing.pairing import pair_emit_to_s2_batched

from data.EMIT.emit_search import (
    search,
    download_reflectance,
    fetch_emit_umm_by_granuleur,
    refetch_emit_pick,
)
from data.EMIT.emit_utils import (
    load_emit_wavelengths_nm_from_nc,
    cache_wavelengths_json,
)
from data.S2.s2_search import (
    download_s2_spectral_stack,
    fetch_s2_item_by_id,
)

from geometry.EMIT_proj import convert_emit_nc_to_envi, crop_s2_stack_to_te
from geometry.crop_utils import (
    valid_bbox_in_map_coords,
    intersect_bounds,
    snap_bounds_to_grid,
    gdal_crop_projwin,
)
from geometry.arosics_coreg import (
    coregister_s2_granule_to_emit_granule,
    s2_bandmap_from_template,
)

from viz.plots import plot_s2_truecolor_from_stack, show_emit_rgb_from_envi

from tiles_helpers.utils import (
    find_valid_paired_tiles,
    save_tile_pair,
    plot_tile_pair_simple,
    write_emit_b32_tile,
)

from documentation.pairs_artifacts import (
    RunPaths,
    ReportWriter,
    TileRecord,
    write_emit_metadata,
    write_s2_metadata,
    write_tile_metadata,
    write_manifest_csv,
    write_archive_map,
    tif_geo_summary,
    write_json,
    copy_any,
    describe_tif,
)

print("All imports OK.")

## 2. Authentication

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

In [ ]:
# Login to https://urs.earthdata.nasa.gov/
ea.login(persist=True)

## 3. Global parameters & output paths

In [ ]:
# ── Region of interest ────────────────────────────────────────────────────
LAT = 36.10
LON = -112.11
SEARCH_BUFFER_M = 5_000

# ── Sentinel-2 STAC ──────────────────────────────────────────────────────
S2_API        = "https://earth-search.aws.element84.com/v1"
S2_COLLECTION = "sentinel-2-l2a"

# ── Quality filters ───────────────────────────────────────────────────────
MAX_S2_CLOUD_FRAC  = 0.7
MAX_EMIT_CLOUD_PCT = 70.0

# ── Output directories ────────────────────────────────────────────────────
DRIVE_ROOT = Path("/content/drive/MyDrive/EMIT_S-2_Matches")
ROOT       = Path("pairs_output")

EMIT_DIR     = ROOT / "emit"
S2_DIR       = ROOT / "s2"
EMIT_UTM_DIR = ROOT / "emit_utm"
FIG_DIR      = ROOT / "plots"
TILES_DIR    = ROOT / "tiles"

for p in (EMIT_DIR, S2_DIR, EMIT_UTM_DIR, FIG_DIR, TILES_DIR):
    p.mkdir(parents=True, exist_ok=True)

print("Output root:", ROOT)

## 4. Find cloud-free EMIT / S2 pairs

In [ ]:
aoi_geom = point_buffer_bbox(LON, LAT, SEARCH_BUFFER_M)

picks = search(
    bbox=aoi_geom.bounds,
    start=None,
    end=None,
    cloud_cover=(0.0, 100.0),
)
print(f"{len(picks)} EMIT granules found in AOI.")

In [ ]:
run_summary = pair_emit_to_s2_batched(
    emit_items       = picks,
    aoi_geom_wgs84   = aoi_geom,
    out_dir          = "/content/s2cloud_runs/PAIRING_RUN_003",
    s2_api           = S2_API,
    s2_collection    = S2_COLLECTION,
    days             = 3.0,
    window_days      = 14,
    resume           = True,
    emit_top_n_per_day   = 5,
    emit_max_cloud_pct   = None,
    scl_cloud_max        = MAX_S2_CLOUD_FRAC,
    top_k_prefilter      = 50,
    tile_m           = 6000.0,
    sun_deg_max      = 5.0,
    max_tod_diff_h   = 1.5,
)
pprint(run_summary)

In [ ]:
import glob
from collections import Counter

base    = "/content/s2cloud_runs/PAIRING_RUN_003"
reasons = Counter()
kept    = 0
total   = 0

pairs = []
for fn in sorted(glob.glob(f"{base}/pairs_batch_*.jsonl")):
    with open(fn) as f:
        for line in f:
            total += 1
            rec = json.loads(line)
            if rec.get("s2_id") and rec.get("s2_cloud_frac") is not None:
                kept += 1
                pairs.append(rec)
            else:
                dbg = rec.get("debug") or {}
                reasons[dbg.get("reason", "unknown")] += 1

print(f"Total candidates: {total}  |  Kept: {kept}")
print("\nRejection reasons:")
for k, v in reasons.most_common():
    print(f"  {k:<45s}  {v}")

## 5. Download one pair

In [ ]:
pair = pairs[1]   # change index to select a different pair
pprint(pair)

In [ ]:
s2_item  = fetch_s2_item_by_id(pair["s2_id"], stac_api=S2_API, collection=S2_COLLECTION)
emit_pick = refetch_emit_pick(pair["emit_granuleur"])

emit_paths = download_reflectance(emit_pick, str(EMIT_DIR), assets=["_RFL_"])
emit_nc    = Path(emit_paths[0])
s2_stack   = download_s2_spectral_stack(s2_item, S2_DIR)

print("EMIT:", emit_nc)
print("S2:  ", s2_stack)

## 6. Preprocessing — metadata, paths, report

In [ ]:
wl_nm   = load_emit_wavelengths_nm_from_nc(str(emit_nc))
wl_json = Path(emit_nc).with_suffix(".wavelengths.json")
cache_wavelengths_json(wl_nm, wl_json)
print(f"{len(wl_nm)} EMIT bands — {wl_nm[0]:.1f}–{wl_nm[-1]:.1f} nm")

In [ ]:
paths  = RunPaths.build(emit_nc=emit_nc, local_root=ROOT, drive_base=DRIVE_ROOT)
report = ReportWriter(paths.drive_report_md, mode="overwrite").start(
    title=f"EMIT and S2 pair: {paths.run_id}"
)
report.section("Run", [
    f"run_id: {paths.run_id}",
    f"local_root: {paths.local_root}",
    f"drive_root: {paths.drive_root}",
])
print("run_id:", paths.run_id)

In [ ]:
emit_summary = write_emit_metadata(emit_pick, paths.drive_meta, report=report)
s2_summary   = write_s2_metadata(s2_item,   paths.drive_meta, report=report)

bounds_wgs84   = emit_summary["spatial"]["bounds_wgs84"]
centroid_wgs84 = emit_summary["spatial"]["centroid_wgs84"]
print("EMIT centroid:", centroid_wgs84)
print("S2 id:        ", s2_summary["id"])

## 7. Convert, crop, co-register, and trim

In [ ]:
# Orthorectify EMIT → UTM-aligned ENVI + GeoTIFF
emit_info_path = paths.drive_emit_utm / "emit_conversion.json"

envi_bin, emit_conv_info = convert_emit_nc_to_envi(
    emit_nc_paths   = [emit_nc],
    s2_visual_path  = s2_stack,
    out_dir         = EMIT_UTM_DIR,
    emit_obs_nc     = None,
    export_loc      = True,
    overwrite       = True,
    return_info     = True,
    save_info_path  = emit_info_path,
    save_geotiffs   = True,
)

print("ENVI bin :", envi_bin)
print("Ortho TIF:", emit_conv_info["outputs"].get("data_ortho_tif"))
print("Warp cmd :", emit_conv_info["commands"]["gdalwarp_data"]["cmd_str"])

report.section("EMIT conversion (orthorectify + warp)", [
    f"Info JSON: {emit_info_path}",
    f"Ortho GeoTIFF: {emit_conv_info['outputs'].get('data_ortho_tif')}",
    f"Warped GeoTIFF: {emit_conv_info['outputs'].get('data_utm_tif')}",
    f"gdalwarp cmd: `{emit_conv_info['commands']['gdalwarp_data']['cmd_str']}`",
])

In [ ]:
# Crop S2 to the EMIT footprint extent
te = emit_conv_info["commands"]["gdalwarp_data"]["aligned_extent"]

s2_overlap, s2_overlap_info = crop_s2_stack_to_te(
    s2_stack_path = s2_stack,
    out_path      = S2_DIR / s2_stack.name.replace("S2_10band_10m", "overlap", 1),
    left          = te["left"],
    bottom        = te["bottom"],
    right         = te["right"],
    top           = te["top"],
    overwrite     = True,
    return_info   = True,
)

s2_overlap_info["geo"] = tif_geo_summary(s2_overlap)
s2_overlap_meta_path   = paths.drive_meta / "s2_overlap_crop.json"
write_json(s2_overlap_meta_path, s2_overlap_info)
copy_any(s2_overlap, paths.drive_s2 / Path(s2_overlap).name, overwrite=False)

report.section("S2 overlap crop", [
    f"Output: {Path(s2_overlap).name}",
    f"Extent (-te): {te}",
    f"Window: {s2_overlap_info.get('window')}",
    f"Geo: {s2_overlap_info['geo'].get('crs')} {s2_overlap_info['geo'].get('size')}",
])
print("S2 overlap:", s2_overlap)

In [ ]:
# Co-register S2 to EMIT using AROSICS tie-point matching
emit_utm_tif = emit_conv_info["outputs"]["data_utm_tif"]
out_s2_tif   = str(Path(s2_overlap).with_name(Path(s2_overlap).stem + "_coreg.tif"))
s2_bandmap   = s2_bandmap_from_template(str(s2_overlap))

res = coregister_s2_granule_to_emit_granule(
    emit_ref_tif    = emit_utm_tif,
    s2_tgt_tif      = s2_overlap,
    out_s2_tif      = out_s2_tif,
    s2_map          = s2_bandmap,
    emit_wl_nm      = wl_nm,
    prefer          = ("B04", "B08"),
    window_size     = (128, 128),
    grid_res        = 150,
    max_points      = 600,
    min_reliability = 60,
    tieP_filter_level = 3,
    max_shift       = 50,
    nodata_emit     = 65535,
    nodata_s2       = 0,
    out_gsd         = [10, 10],
    cliptoextent    = False,
)

status = "OK" if res["final"]["success"] else "FAIL"
print(f"Co-registration: {status}")
if res["final"]["success"]:
    print("Output:", res["out_s2_tif"])
else:
    print("Attempts:", res["attempts"])

In [ ]:
# Trim both images to their valid-pixel intersection, snapped to a 60 m grid
TE_overlap    = (te["left"], te["bottom"], te["right"], te["top"])
x0, y0        = te["anchor_x0"], te["anchor_y0"]

valid_te = valid_bbox_in_map_coords(out_s2_tif, margin_px=2)
te_trim  = intersect_bounds(TE_overlap, valid_te)
te_trim  = snap_bounds_to_grid(te_trim, x0=x0, y0=y0, grid_m=60.0)

out_s2_tif_trimmed = str(
    Path(s2_overlap).with_name(Path(s2_overlap).stem + "_coreg_trimmed.tif")
)
envi_bin_trimmed = str(envi_bin.with_name(envi_bin.stem + "_trimmed.bin"))

gdal_crop_projwin(out_s2_tif,    out_s2_tif_trimmed, te_trim, out_format="GTiff")
gdal_crop_projwin(str(envi_bin), envi_bin_trimmed,   te_trim, out_format="ENVI",
                  extra=["-co", "INTERLEAVE=BIL"])

print("Trimmed S2:  ", out_s2_tif_trimmed)
print("Trimmed EMIT:", envi_bin_trimmed)

## 8. Visualisation

In [ ]:
# S2 vs EMIT after orthorectification (before trimming)
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))
plot_s2_truecolor_from_stack(str(s2_overlap), ax=ax1)
show_emit_rgb_from_envi(
    str(EMIT_UTM_DIR),
    pattern=str(envi_bin.name),
    wavelengths_nm=wl_nm,
    ax=ax2,
    gamma=1 / 2.2,
)
plt.tight_layout()
plt.show()

In [ ]:
# S2 vs EMIT after co-registration and trimming
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))
plot_s2_truecolor_from_stack(out_s2_tif_trimmed, ax=ax1)
show_emit_rgb_from_envi(
    str(EMIT_UTM_DIR),
    pattern=str(Path(envi_bin_trimmed).name),
    wavelengths_nm=wl_nm,
    ax=ax2,
    gamma=1 / 2.2,
)
plt.tight_layout()
plt.show()

## 9. Tiling and export

In [ ]:
TILE_PARAMS = {"emit_tile_size": 100, "scale": 6, "max_black_frac": 0.0}

valid_tiles = find_valid_paired_tiles(
    emit_path = str(envi_bin_trimmed),
    s2_path   = str(out_s2_tif_trimmed),
    **TILE_PARAMS,
)
print(f"{len(valid_tiles)} valid tile positions found.")

In [ ]:
tile_records: list[TileRecord] = []
emit_b32_idx: np.ndarray | None = None
wrote_b32_report = False

for tile_info in tqdm(valid_tiles, desc="Tiles"):
    k = int(tile_info["idx"])

    # ── save raw tile pair ────────────────────────────────────────────────
    emit_out, s2_out = save_tile_pair(
        emit_path = str(envi_bin_trimmed),
        s2_path   = str(out_s2_tif_trimmed),
        tile_info = tile_info,
        out_dir   = paths.drive_tiles,
    )

    # ── 32-band EMIT subset ───────────────────────────────────────────────
    emit_out_b32, emit_b32_idx = write_emit_b32_tile(
        emit_out,
        num_keep  = 32,
        idx_0based = emit_b32_idx,
        overwrite = True,
    )
    if not wrote_b32_report:
        report.section("EMIT 32-band subset", [
            "Per tile we save *_b32.tif with 32 evenly-spaced EMIT bands.",
            f"Indices (0-based): {emit_b32_idx.tolist()}",
        ])
        wrote_b32_report = True

    # ── plot ──────────────────────────────────────────────────────────────
    plot_png = paths.drive_plots / f"tile_{k:03d}_pair.png"
    plot_tile_pair_simple(
        emit_tile_path     = emit_out,
        s2_tile_path       = s2_out,
        title_suffix       = f"{k:03d}",
        save_path          = plot_png,
        show               = True,
        emit_wavelengths_nm = wl_nm,
    )

    # ── record ────────────────────────────────────────────────────────────
    rec = TileRecord(
        idx             = k,
        emit_tif        = str(emit_out),
        s2_tif          = str(s2_out),
        plot_png        = str(plot_png),
        emit_black_frac = tile_info.get("emit_black_frac"),
        s2_black_frac   = tile_info.get("s2_black_frac"),
    )
    rec.emit_geo = tif_geo_summary(emit_out)
    rec.s2_geo   = tif_geo_summary(s2_out)
    rec.emit_b32_tif = str(emit_out_b32)
    rec.emit_b32_indices_0based = emit_b32_idx.tolist()

    ew = tile_info["emit_window"]
    sw = tile_info["s2_window"]
    rec.emit_window = {"row_off": int(ew.row_off), "col_off": int(ew.col_off),
                       "height": int(ew.height),  "width":  int(ew.width)}
    rec.s2_window   = {"row_off": int(sw.row_off), "col_off": int(sw.col_off),
                       "height": int(sw.height),  "width":  int(sw.width)}

    write_tile_metadata(
        rec,
        tile_info    = tile_info,
        out_dir      = paths.drive_tile_meta,
        emit_granule = emit_summary.get("granule_ur"),
        emit_time    = emit_summary.get("time"),
        s2_id        = s2_summary.get("id"),
        s2_datetime  = s2_summary.get("datetime"),
        params       = TILE_PARAMS,
    )

    tile_records.append(rec)
    print("Saved:", Path(rec.emit_tif).name, Path(rec.s2_tif).name)

write_manifest_csv(paths.drive_manifest_csv, tile_records)
print(f"\nManifest: {paths.drive_manifest_csv}  ({len(tile_records)} tiles)")

In [ ]:
example_centroid = None
if tile_records and isinstance(tile_records[0].s2_geo, dict):
    example_centroid = tile_records[0].s2_geo.get("centroid_wgs84")

report.section("Tiles summary", [
    f"Number of saved tiles: {len(tile_records)}",
    f"Example tile centroid (S2, WGS84): {example_centroid}" if example_centroid else None,
    f"Tile metadata folder: {paths.drive_tile_meta}",
    f"Manifest: {paths.drive_manifest_csv}",
])

## 10. Archive to Drive and finalise report

In [ ]:
copy_any(
    paths.local_emit_utm / "geotiff",
    paths.drive_emit_utm,
    overwrite=True,
    exclude=["tmp", "*.aux.xml", "*.ovr"],
)
copy_any(emit_nc,  paths.drive_emit / Path(emit_nc).name,  overwrite=True)
copy_any(s2_stack, paths.drive_s2   / Path(s2_stack).name, overwrite=True)

archive_map = {
    "local_ROOT":                str(ROOT),
    "drive_pair_folder":         str(DRIVE_ROOT),
    "drive_raw_emit":            str(paths.drive_emit),
    "drive_raw_s2":              str(paths.drive_s2),
    "drive_emit_reprojections":  str(paths.drive_emit_utm),
}
write_archive_map(paths.drive_meta / "archive_map.json", archive_map, report=report)
print("Archive complete.  Report:", paths.drive_report_md)

In [ ]:
# Inspect final outputs
describe_tif(paths.drive_emit_utm / f"L2A_RFL_{paths.run_id}_DATA_warp_utm.tif")
describe_tif(paths.drive_emit_utm / f"L2A_RFL_{paths.run_id}_DATA_ortho_wgs84.tif")
if tile_records:
    describe_tif(paths.drive_tiles / f"tile_{tile_records[0].idx:03d}_s2.tif")